# Open API를 활용한 네이버 뉴스 검색

## 1. 애플리케이션 등록
https://developers.naver.com/apps/#/register

## 2. 환경변수 관리
- 등록된 애플리케이션 페이지에서 제공되는 Client ID, Secret은 절대 외부로 노출되면 안된다
- dotenv (.env)를 통해서 관리
    - `pip install python-dotenv` 패키지 설치
    - .gitignore 파일에 .env 무시하는 구문 추가


In [1]:
import http
!pip install python-dotenv

### 프로젝트 하위 폴더에 .env 파일을 만들어 아래 값 입력
```
NAVER_CLINET_ID = {{Client Id}}
NAVER_CLIENT_SECRET = {{Client Secret}}
```
- 네이버 개발자센터에 등록된 애플리케이션에서 확인 가능

In [3]:
# .env 파일을 로드해서 환경변수로 등록
from dotenv import load_dotenv
load_dotenv()   # 같은 경로상에 있는 .env 파일을 읽어와 자동으로 환경변수로 등록
# 읽어오기 성공 시 True, 실패 시 False 반환

True

In [9]:
# 환경변수에서 .env 등록한 내용 얻어오기
import os
NAVER_CLIENT_ID = os.getenv('NAVER_CLINET_ID')
NAVER_CLIENT_SECRET = os.getenv('NAVER_CLIENT_SECRET')

# api key, 비밀번호를 print 하는 구문을 실수로 남겨놓으면 노출될 가능성 있음
# -> jupyter 변수 탭에서 확인

# .env 내용이 환경변수로 등록되지 않은 경우
if not NAVER_CLIENT_ID or not NAVER_CLIENT_SECRET:
    raise ValueError('네이버 클라이언트 아이디 또는 시크릿이 환경변수에 등록되지 않았습니다.')

### 3. API 요청
- 파이썬에서 웹 요청을(http)을 처리하기 위해서는 `requests` 라이브러리 필요

`!pip install requests`

In [5]:
!pip install requests

In [10]:
# 네이버 뉴스 검색 API 요청
import urllib.request
import socket

encText = urllib.parse.quote('인공지능') # url encoding 작업
# 한글 -> %시작하는 문자로 변경

url = f'https://openapi.naver.com/v1/search/news.json?query={encText}&display=10&sort=date'
# 쿼리스트링 == 전달할 값(파라미터)가 작성된 문자열

# 요청 객체 생성
request = urllib.request.Request(url)
# Header: 주소, 요청정보 / Body: 파라미터

# API 인증 정보를 요청 헤더에 추가
request.add_header("X-Naver-Client-Id", NAVER_CLINET_ID)
request.add_header("X-Naver-Client-Secret", NAVER_CLIENT_SECRET)

try:
    with urllib.request.urlopen(request, timeout=10) as response:
        # 지정된 주소로 요청 -> 결과를 response로 전달 받음
        # 단, 요청 대기시간이 10초를 초과하면 중지

        # HTTP 응답 상태 코드. 200이면 정상 응답 == 성공
        response_code = response.getcode()

        # 응답 본문 확인 (bytes - UTF-8로 변환)
        response_body = response.read().decode('utf-8')

        print("response_code: ", response_code)
        print(response_body)
        # 응답 본문이 JSON(str 타입) 형태
        # -> 이용이 필요할 경우 parsing 작업 필수

except socket.timeout:
    print("요청 시간 10초 초과")

response_code:  200
{
	"lastBuildDate":"Mon, 15 Jun 2026 11:40:03 +0900",
	"total":4045728,
	"start":1,
	"display":10,
	"items":[
		{
			"title":"&quot;인간 창작자 권익 지킨다&quot; 쿠키플레이스, 생성형 AI 활용 금지 원칙 발...",
			"originallink":"http:\/\/www.inews24.com\/view\/1976500",
			"link":"https:\/\/n.news.naver.com\/mnews\/article\/031\/0001035021?sid=105",
			"description":"새 약관에 따르면 쿠키플레이스는 생성형 AI를 '이용자의 입력(프롬프트)에 따라 텍스트·이미지·음악·영상 등 새로운 콘텐츠를 자동으로 생성하는 <b>인공지능<\/b> 시스템'으로 정의했다. 일반적인 콘텐츠 외주와 달리... ",
			"pubDate":"Mon, 15 Jun 2026 11:38:00 +0900"
		},
		{
			"title":"삼성전자, 제품 개발도 AI 전환…검증기간 15일서 2일로 단축",
			"originallink":"http:\/\/www.4th.kr\/news\/articleView.html?idxno=2113175",
			"link":"http:\/\/www.4th.kr\/news\/articleView.html?idxno=2113175",
			"description":"포쓰저널 송신용 기자 삼성전자 서초사옥 \/ 사진=연합 삼성전자가 스마트폰과 가전 제품 개발 과정에 고성능컴퓨팅(HPC) 인프라를 도입하며 <b>인공지능<\/b>(AI) 기반 개발 체계 고도화에 나섰다. 실제 제품을 만들기 전 가상... ",
			"pubDate":"Mon, 15 Jun 2026 11:38:00 +0900"
		},
		{
			"title":"영원무역홀딩스, 한국패션비즈니스학회와 국제 패션전 개최…

In [14]:
# requests 객체를 이용한 요청(더 쉬움)
import requests
from pprint import pprint   # 출력 시 공백문자를 이용해 가독성 좋게 출력

url = 'https://openapi.naver.com/v1/search/news.json'

# Header, Body에 전달할 값을 dict 형식으로 생성
headers = {
    "X-Naver-Client-Id": NAVER_CLIENT_ID,
    "X-Naver-Client-Secret": NAVER_CLIENT_SECRET
}

prarms = {
    'query' : '인공지능',
    'display' : 10,
    'start' : 1,
    'sort' : 'date'
}

try:
    # GET Method == 조회 요청
    response = requests.get(
        url,
        headers=headers,
        params=prarms,  # dict -> 쿼리 스트링 변환(+ url encoding)
        timeout=10
    )

    # HTTP 상태 코드가 400, 500번대인 경우 예외 발생
    response.raise_for_status()

    response_code = response.status_code    # 상태 코드
    data = response.json()  # 응답 데이터(json) -> dict 변환

    print('response_code: ', response_code)
    #pprint(data)

    pprint(data['items'][0])

except requests.exceptions.Timeout:
    print("요청 시간 10초 초과")
except ValueError:
    print("응답 데이터가 JSON 형식이 아닙니다")

response_code:  200
{'description': '이 분야는 연구·개발(R&amp;D)과 건축 엔지니어링, 법률·회계 서비스 등이 포함된 업종이라 '
                '<b>인공지능</b>(AI) 도입의 영향을 받았을 가능성이 제기된다. 정부는 AI가 채용 위축에 미친 영향을 '
                '단정하기는 이르다는... ',
 'link': 'https://n.news.naver.com/mnews/article/021/0002797757?sid=101',
 'originallink': 'https://www.munhwa.com/article/11595580?ref=naver',
 'pubDate': 'Mon, 15 Jun 2026 11:49:00 +0900',
 'title': '제조업 부진에… 26년5개월만에 상용직 줄었다'}


위에 셀 모듈로 만들어보기

In [20]:
# requests 객체를 이용한 요청(더 쉬움)
import requests
from pprint import pprint   # 출력 시 공백문자를 이용해 가독성 좋게 출력

url = 'https://openapi.naver.com/v1/search/news.xml'

# Header, Body에 전달할 값을 dict 형식으로 생성
headers = {
    "X-Naver-Client-Id": NAVER_CLIENT_ID,
    "X-Naver-Client-Secret": NAVER_CLIENT_SECRET
}

prarms = {
    'query' : '인공지능',
    'display' : 10,
    'start' : 1,
    'sort' : 'date'
}

try:
    # GET Method == 조회 요청
    response = requests.get(
        url,
        headers=headers,
        params=prarms,  # dict -> 쿼리 스트링 변환(+ url encoding)
        timeout=10
    )

    # HTTP 상태 코드가 400, 500번대인 경우 예외 발생
    response.raise_for_status()

    response_code = response.status_code    # 상태 코드
    data = response.content  # 응답 문자열(xml)

    # XML 해석 파이썬 기본 내장 모듈
    # -> xml.etree.ElementTree

    # XML 파일로 저장
    with open('response.xml', 'wb') as f:
        f.write(data)

    print("response.xml 저장 완료")

except requests.exceptions.Timeout:
    print("요청 시간 10초 초과")


response.xml 저장 완료
